# Overture Streets — Edge Bearings (Tahoe Basin Trails)

Same approach as `Overture_EdgeBearings.ipynb` (roads) but filtered to
non-motorised trail classes: path, track, footway, cycleway, bridleway.

Steps:
1. Load segments from the GeoParquet saved by `OvertureStreets_Download.ipynb`
2. Spatial filter to the Tahoe Basin only (excludes Truckee + Carson City)
3. Filter to trail classes only
4. `add_edge_bearings()` — vectorised equivalent of `osmnx.add_edge_bearings()`
5. Histogram of edge bearings
6. Polar orientation plot (compass rose)

In [ ]:
import os
import pyproj

# ArcGIS Pro conda env: fix PROJ data directory at runtime
_proj_data = r"C:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Library\share\proj"
pyproj.datadir.set_data_dir(_proj_data)
try:
    _ = pyproj.CRS("EPSG:4326")
    print(f"PROJ OK — data dir: {pyproj.datadir.get_data_dir()}")
except Exception as e:
    raise

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Overture release — must match the parquet produced by OvertureStreets_Download.ipynb
OVERTURE_RELEASE = '2026-05-20.0'

SEG_PARQUET = Path('.') / f'tahoe_overture_segments_{OVERTURE_RELEASE}.parquet'

# Tahoe Basin bbox (WGS 84: minx, miny, maxx, maxy)
# Tighter than the download bbox; excludes Truckee (north) and Carson City (east)
BASIN_BBOX = (-120.15, 38.89, -119.90, 39.27)

# Non-motorised trail classes (excludes steps — those are infrastructure, not trails)
TRAIL_CLASSES = {
    'path', 'track', 'footway', 'cycleway', 'bridleway',
}

print(f"Parquet : {SEG_PARQUET}")
print(f"Basin   : {BASIN_BBOX}")
print(f"Classes : {sorted(TRAIL_CLASSES)}")

### Load Overture segments and filter to trails

In [ ]:
segments_gdf = gpd.read_parquet(SEG_PARQUET)
print(f"Loaded   : {len(segments_gdf):,} segments (full download bbox)")

# Clip to Tahoe Basin using the spatial index (.cx accessor)
basin_gdf = segments_gdf.cx[
    BASIN_BBOX[0]:BASIN_BBOX[2],
    BASIN_BBOX[1]:BASIN_BBOX[3]
].copy()
print(f"Basin    : {len(basin_gdf):,} segments after spatial filter")

# Trail network only
G = basin_gdf[basin_gdf['class'].isin(TRAIL_CLASSES)].copy()
print(f"Trails   : {len(G):,} segments")
print()
print(G['class'].value_counts().to_string())

### Trail network map

Equivalent of `ox.plot_graph(ox.project_graph(G), node_size=0)`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
G.to_crs('EPSG:3857').plot(ax=ax, color='k', linewidth=0.3)
ax.set_aspect('equal')
ax.set_axis_off()
ax.set_title('Tahoe Basin Trails', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('tahoe_trail_network.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### Calculate edge bearings

Vectorised replication of `osmnx.add_edge_bearings()`: forward azimuth from the
first to last coordinate of each LineString using the geodesic formula on WGS 84.

In [ ]:
def add_edge_bearings(gdf):
    """
    Add a 'bearing' column (0-360 deg, clockwise from north) to a LineString GeoDataFrame.
    Mirrors osmnx.add_edge_bearings() without requiring a networkx graph.
    """
    lat1 = gdf.geometry.apply(lambda g: g.coords[0][1]).values
    lon1 = gdf.geometry.apply(lambda g: g.coords[0][0]).values
    lat2 = gdf.geometry.apply(lambda g: g.coords[-1][1]).values
    lon2 = gdf.geometry.apply(lambda g: g.coords[-1][0]).values

    lat1_r = np.radians(lat1)
    lat2_r = np.radians(lat2)
    dlon_r = np.radians(lon2 - lon1)

    x = np.sin(dlon_r) * np.cos(lat2_r)
    y = (np.cos(lat1_r) * np.sin(lat2_r)
         - np.sin(lat1_r) * np.cos(lat2_r) * np.cos(dlon_r))

    bearing = np.degrees(np.arctan2(x, y))
    gdf = gdf.copy()
    gdf['bearing'] = (bearing + 360) % 360
    return gdf


G = add_edge_bearings(G)
print(f"Bearings added. Range: {G['bearing'].min():.1f} deg - {G['bearing'].max():.1f} deg")
G[['class', 'name', 'bearing']].head(5)

In [ ]:
# Extract as a Series — mirrors the OSMnx pattern:
#   pd.Series([data['bearing'] for u, v, k, data in G.edges(keys=True, data=True)])
bearings = G['bearing']

ax = bearings.hist(bins=30, zorder=2, alpha=0.8, color='#555555')
ax.set_xlim(0, 360)
ax.set_xticks([0, 90, 180, 270, 360])
ax.set_xticklabels(['N (0)', 'E (90)', 'S (180)', 'W (270)', 'N (360)'])
ax.set_title('Tahoe Basin Trails — edge bearing distribution')
plt.show()

In [ ]:
# Polar orientation plot — same style as Overture_EdgeBearings.ipynb
n = 30
count, division = np.histogram(bearings, bins=[ang * 360 / n for ang in range(0, n + 1)])
division = division[0:-1]
width = 2 * np.pi / n

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={'projection': 'polar'})
ax.set_theta_zero_location('N')
ax.set_theta_direction('clockwise')
ax.bar(
    division * np.pi / 180 - width * 0.5, count,
    width=width, bottom=0.0,
    color='#555555', edgecolor='white', linewidth=0.5, alpha=0.9
)
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2])
ax.set_xticklabels(['N', 'E', 'S', 'W'], fontsize=14, fontweight='bold')
ax.set_yticklabels([])
ax.set_title('Tahoe Basin Trails', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('tahoe_bearing_polar_trails.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()